# WaferScan · GPU training on Google Colab or Kaggle

This notebook trains the **big backbones** the CPU can't: EfficientNet-B7, SE-ResNeXt-101 (ResNeXt + attention) and Swin-T (a Vision Transformer with hierarchical patch merging). Then it **stacks** them with the small WaferNet and the morphology model, and exports ONNX files you can drop into the API.

**Before you run:**
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2**, and turn **Internet ON** (phone-verified account). Add the dataset *"WM-811K wafer map"* (qingyi/wm811k-wafer-map) with **+ Add Input**.

**Mentor note: why checkpoints matter here.** Free GPUs disconnect. Every epoch of every fold is saved to disk. After a disconnect, run the cells again from the top: finished folds are skipped, and a half-finished fold resumes at its last epoch.

Rough time per fold on a T4: wafernet ~6 min · swin_t ~26 min · seresnext101 ~50 min · efficientnet_b7 ~85 min. Use `FOLDS` below to spread the work over several sessions.

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/waferscan_work"   # on Drive = survives disconnects
elif ON_KAGGLE:
    WORK = "/kaggle/working"                          # kept as notebook output
else:
    WORK = os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
print("Colab:", ON_COLAB, "| Kaggle:", ON_KAGGLE, "| work dir:", WORK)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Get the code
Either clone your GitHub repo (after you push it), or upload `waferscan.zip` (to Drive on Colab, or as a Kaggle dataset) and point `ZIP` at it.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<you>/waferscan.git"
ZIP = ""        # e.g. "/content/drive/MyDrive/waferscan.zip" or "/kaggle/input/waferscan-code/waferscan.zip"
%cd {WORK}
if not os.path.exists("waferscan"):
    if REPO_URL:
        !git clone -q {REPO_URL} waferscan
    elif ZIP:
        !unzip -q -o {ZIP} -d .
    else:
        raise SystemExit("set REPO_URL or ZIP first")
%cd {WORK}/waferscan
!pip install -q -e ".[api,train,dev]"

## 2 · Data
**Option A (Kaggle, recommended):** the full WM-811K pickle: 811,457 wafers, of which ~173k are labelled. We keep every labelled defect wafer and a random 20k of the 147k *none* wafers. Groups = **lots**, so a lot never appears in both training and validation.

**Option B:** the 6,309-image curated set already in the repo (`data/processed/rendered_wm811k.npz`).

In [ ]:
LSWMD = "/kaggle/input/wm811k-wafer-map/LSWMD.pkl"   # Colab: e.g. "/content/drive/MyDrive/LSWMD.pkl"
DATA = f"{WORK}/wm811k.npz"
if os.path.exists(LSWMD) and not os.path.exists(DATA):
    !python -m waferscan.data.wm811k --pkl {LSWMD} --out {DATA} --none-cap 20000
if not os.path.exists(DATA):
    DATA = "data/processed/rendered_wm811k.npz"
print("training data:", DATA)

## 3 · Train the backbones (5-fold grouped cross-validation each)
Each run logs to MLflow (a database file in your work dir). Remove the slow backbones from the list if you're short on time; the stacking step uses whatever finished.

In [ ]:
RUNS = f"{WORK}/runs"
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{WORK}/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
BACKBONES = ["wafernet", "swin_t", "seresnext101", "efficientnet_b7"]
FOLDS = ""        # e.g. "0 1 2" this session, "3 4" next session; "" = all five
for bb in BACKBONES:
    extra = f"--folds {FOLDS}" if FOLDS else ""
    !python -m waferscan.training.train --config configs/gpu_full.yaml --backbone {bb} --data {DATA} --out {RUNS}/{bb} {extra}

## 4 · Stack, calibrate, export
Stacking needs **all five folds** of a backbone (its out-of-fold predictions cover every wafer). The meta-learner learns how much to trust each backbone and the morphology model. The model card records every metric, per backbone and stacked.

In [ ]:
import yaml
done = [f"{RUNS}/{bb}" for bb in BACKBONES
        if os.path.exists(f"{RUNS}/{bb}/oof.npz")
        and all(os.path.exists(f"{RUNS}/{bb}/fold{k}.pt") for k in range(5))]
print("complete runs:", done)
BUNDLE = f"{WORK}/model_bundle"
runs = " ".join(done)
!python -m waferscan.training.stack --runs {runs} --data {DATA} --bundle {BUNDLE}
!python -m waferscan.export.onnx_export --runs {runs} --data {DATA} --bundle {BUNDLE}

In [ ]:
import json, pandas as pd
card = json.load(open(f"{BUNDLE}/model_card.json"))
table = {k: {"accuracy": v["accuracy"], "macro_F1": v["macro"]["f1"], "MCC": v["mcc"],
             "kappa": v["cohen_kappa"], "top2": v["top2_accuracy"], "ECE": v["ece"]}
         for k, v in card["metrics"].items() if not k.endswith(("_mc_dropout", "_evidential"))}
print("meta-learner weight share:", card["meta_weight_share"])
print("routing (out-of-fold):", {k: card["routing_oof"][k] for k in ("coverage_auto_accepted", "accuracy_auto_accepted")})
pd.DataFrame(table).T.sort_values("macro_F1", ascending=False).round(4)

## 5 · Take the bundle home
Download `model_bundle.zip` and unzip it over the repo's `model_bundle/` folder. The API, the landing page and the Triton/Kubernetes deployment all read from there.

In [ ]:
!cd {WORK} && rm -f model_bundle.zip && zip -qr model_bundle.zip model_bundle && ls -lh model_bundle.zip
if ON_COLAB:
    from google.colab import files
    files.download(f"{WORK}/model_bundle.zip")

## Mentor corner: what to look at, and 3 exercises
1. **Backbone table.** Is the 66M-parameter EfficientNet-B7 actually better than the 0.5M WaferNet on 64×64 die maps? Big models help when inputs are large and detailed. A wafer map is small and blocky, so don't assume. *Exercise:* note accuracy, macro-F1 and training time per backbone. Which gives the best F1 per GPU-minute?
2. **Per-class F1.** On the full WM-811K, look at *Loc* vs *Edge-Loc* and *Scratch*. *Exercise:* open the confusion matrix and find the most confused pair. Look at 5 of those wafers yourself: would *you* agree with the labels?
3. **Routing.** `coverage_auto_accepted` = the share of wafers the system decides alone at ≥99.5% accuracy. *Exercise:* in `waferscan/training/stack.py`, change `AUTO_ACCEPT_TARGET` to 0.99 and to 0.999, re-run cell 4 and plot coverage against target. This trade-off is what "zero misclassification" really means in production.